In [0]:
import dlt
from pyspark.sql.functions import *
from pyspark.sql.window import Window

In [0]:
@dlt.table(
    name="gold.driver_standings",
    comment="Classificação do campeonato de pilotos"
)
def gold_driver_standings():
    """
    Agregação para classificação de pilotos
    """
    return (
        dlt.read("silver.results")
        .groupBy("season", "driver_id", "driver_name")
        .agg(
            sum("points").alias("total_points"),
            count("race_id").alias("races_participated"),
            sum(when(col("final_position") == 1, 1).otherwise(0)).alias("wins"),
            sum(when(col("final_position") <= 3, 1).otherwise(0)).alias("podiums"),
            avg("final_position").alias("avg_finish_position")
        )
        .withColumn(
            "rank",
            row_number().over(Window.partitionBy("season").orderBy(desc("total_points")))
        )
    )